# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [48]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 250
SPATIAL_UNIT = "community" # options: census, hexa, community

In [49]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from joblib import dump, load

## Preparations

In [50]:
# "Settings" / Decisions for the training data
if SPATIAL_UNIT == "census":    
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "hexa":
    DATA_PATH_TRAIN = "../data/train_test_data/svm_hexa_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_hexa_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_hexa_test.parquet" 
elif SPATIAL_UNIT == "community": 
    DATA_PATH_TRAIN = "../data/train_test_data/svm_community_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_community_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_community_test.parquet" 
else:
    print("Warning: No type of Spatial Unit given, Used census tract")
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 



MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_count"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_demand",
    "date",
]

Load data and select features and target

In [51]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [52]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [53]:
train_df.head()

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
0,2024-01-11 07:00:00,1,4,7,0.0,1.000000,0.433884,-0.900969,0.965926,-2.588190e-01,...,0.0,0.0,0.0,0.0,0.0,87.64,43.82,32.5,55.14,Unknown
1,2024-01-11 02:00:00,1,4,2,0.0,1.000000,0.433884,-0.900969,0.500000,8.660254e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips
2,2024-01-08 11:00:00,1,1,11,0.0,1.000000,0.000000,1.000000,0.258819,-9.659258e-01,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips
3,2024-02-20 06:00:00,2,2,6,0.5,0.866025,0.781831,0.623490,1.000000,6.123234e-17,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips
4,2024-02-16 00:00:00,2,5,0,0.5,0.866025,-0.433884,-0.900969,0.000000,1.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,0.00,No trips


In [54]:
train_df = train_df.sample(n=5000, random_state=42)
train_df_grid = train_df.sample(n=GRID_SAMPLE, random_state=42)

In [55]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]

X_val = val_df[feature_cols]
y_val = val_df[TARGET_COL]

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]


print("Features:", X_train.dtypes)
print("Target:", y_train.dtypes)

Features: month_sin         float64
month_cos         float64
weekday_sin       float64
weekday_cos       float64
hour_sin          float64
hour_cos          float64
tmpc              float64
relh              float64
sknt              float64
vsby              float64
p01m              float64
skyc1_BKN            int8
skyc1_CLR            int8
skyc1_FEW            int8
skyc1_OVC            int8
skyc1_SCT            int8
skyc1_VV             int8
is_holiday           int8
community_area      int64
food_drink        float64
landmark          float64
shop              float64
train_station     float64
dtype: object
Target: uint32


In [56]:
# Create X and y for grid search
X_train_grid = train_df_grid[feature_cols]
y_train_grid = train_df_grid[TARGET_COL]

In [57]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
445134,2024-01-04 12:00:00,1,4,12,0.000000e+00,1.000000e+00,0.433884,-0.900969,1.224647e-16,-1.000000e+00,...,0.0,0.0,0.000000,0.0,0.0,28.75,28.750000,28.75,28.75,Prcard
234150,2026-03-05 22:00:00,3,4,22,8.660254e-01,5.000000e-01,0.433884,-0.900969,-5.000000e-01,8.660254e-01,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
767563,2026-04-06 06:00:00,4,1,6,1.000000e+00,6.123234e-17,0.000000,1.000000,1.000000e+00,6.123234e-17,...,0.0,0.0,0.000000,0.0,0.0,101.64,50.820000,31.00,70.64,Cash
415034,2025-06-08 16:00:00,6,7,16,5.000000e-01,-8.660254e-01,-0.781831,0.623490,-8.660254e-01,-5.000000e-01,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
365383,2024-11-14 23:00:00,11,4,23,-8.660254e-01,5.000000e-01,0.433884,-0.900969,-2.588190e-01,9.659258e-01,...,1.0,28.0,0.571429,0.0,5.0,1005.37,20.517755,4.25,98.25,Cash
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924743,2025-12-01 09:00:00,12,1,9,-5.000000e-01,8.660254e-01,0.000000,1.000000,7.071068e-01,-7.071068e-01,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips
863377,2024-07-02 09:00:00,7,2,9,1.224647e-16,-1.000000e+00,0.781831,0.623490,7.071068e-01,-7.071068e-01,...,0.0,0.0,0.000000,0.0,0.0,118.00,29.500000,21.50,36.25,Unknown
709857,2025-12-24 21:00:00,12,3,21,-5.000000e-01,8.660254e-01,0.974928,-0.222521,-7.071068e-01,7.071068e-01,...,0.0,0.0,0.000000,0.0,0.0,110.87,27.717500,6.25,50.37,Cash
325990,2026-04-15 03:00:00,4,3,3,1.000000e+00,6.123234e-17,0.974928,-0.222521,7.071068e-01,7.071068e-01,...,0.0,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips


In [58]:
model = SVR()

In [59]:
if DO_GRID_SEARCH:
    param_grid = {
        "C": [50, 100, 150],
        "kernel": ["linear", "rbf", "poly", "sigmoid"],
        "gamma": ["scale", "auto", 0.01, 0.1, 1]
    }

    # Perform grid search
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,               
        scoring="r2",  
        n_jobs=-1,
        error_score="raise"
    )

    # Fit
    grid_search.fit(X_train_grid, y_train_grid)


In [60]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 50, 'gamma': 0.01, 'kernel': 'rbf'}
Best CV score: 0.12657895011204606


In [61]:
# dump/load grid search

dump(grid_search, "../models/grid_search_svr.joblib")

['../models/grid_search_svr.joblib']

In [62]:
best_model = grid_search.best_estimator_

In [63]:
best_model

,kernel,'rbf'
,degree,3
,gamma,0.01
,coef0,0.0
,tol,0.001
,C,50
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [64]:
# Train SVR 

best_model.fit(X_train, y_train)

,kernel,'rbf'
,degree,3
,gamma,0.01
,coef0,0.0
,tol,0.001
,C,50
,epsilon,0.1
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [65]:
# Make prediction 
y_pred = best_model.predict(X_test)

In [66]:
y_pred

array([1.25205022, 1.15057516, 1.33177367, ..., 1.89747634, 2.33184301,
       1.44327031], shape=(237252,))

In [67]:
# Evaluation metrics

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 6.387973689916219
MSE: 598.749612509763
RMSE: 24.469360688619616
R2 Score: 0.5077281509250717


In [ ]:
# save model
dump(best_model, "../models/svm_" + SPATIAL_UNIT + "_svr.joblib")

['../models/svmcommunity_svr.joblib']